# Hands-On 2 · Not Fooling Yourself

**Introduction to Machine Learning** · MDMC 2026 · Day 2, Tuesday 10 November · Paolo Muratore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/myscience/intro-ml-mdmc-2026/blob/main/notebooks/N2_not_fooling_yourself.ipynb)

This notebook rebuilds this morning's slides with your own hands: first the U-curve (*The U-Curve*), by fitting polynomials of rising degree and measuring them on data they have not seen; then the honest protocol of *Cross-Validation* and *Turning the Knob λ*, with a scikit-learn pipeline. The last part is a case from *Data Leakage* and *A Curator's Checklist*: a published-looking result that is too good to be true, for you to take apart.

| Tier | What | Time |
|---|---|---|
| ★ Core | Overfitting by hand. Everyone finishes this part. | about 30 min |
| ★★ Stretch | Cross-validation with a pipeline: choose the degree and λ. | about 20 min |
| ★★★ Challenge | Find the leak. | about 20 min |

Blanks look like `____` or `# YOUR CODE HERE`. Each exercise ends with a **checkpoint** that tells you whether you got it right, and a **Question**: answer it in a sentence or two, in the cell itself.

## Setup

Run the next two cells once. The first imports everything we need; the second holds the data-generating process from the slides and the plotting helpers. You do not need to edit either.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from numpy.polynomial import Polynomial

from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (GridSearchCV, GroupKFold, KFold, cross_val_score,
                                     train_test_split, validation_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

SEED = 0                                         # one seed for every random choice: we all see the same numbers

# course palette (light theme)
YELLOW, CYAN, MAGENTA, LIME = "#D99E00", "#0094AE", "#D1006C", "#7E8E00"
INK, GREY = "#111111", "#6B6B6B"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 100, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# Helpers: run this cell, no need to read or edit it.

def f_true(x):
    """The function behind the data, the same as on the slides."""
    return np.sin(2 * np.pi * x)


SIG = 0.3                                        # standard deviation of the noise


def sample(n, seed):
    """n points, x uniform in [0, 1], y = sin(2 pi x) + Gaussian noise of std 0.3; sorted by x."""
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(0, 1, n))
    return x, f_true(x) + rng.normal(0, SIG, n)


XX = np.linspace(0, 1, 500)                      # fine grid of x values, for drawing curves


def _frame(ax, x_train, y_train, x_other=None, y_other=None, other="validation", train_label="training"):
    ax.plot(XX, f_true(XX), color=GREY, ls="--", lw=1.5, label="true function")
    ax.scatter(x_train, y_train, color=INK, s=22, zorder=3, label=train_label)
    if x_other is not None:
        ax.scatter(x_other, y_other, color=LIME if other == "validation" else MAGENTA,
                   marker="D", s=22, zorder=3, label=other)
    ax.set(xlim=(0, 1), ylim=(-2, 2), xlabel="$x$", ylabel="$y$")


def plot_data(x_train, y_train, x_val, y_val):
    fig, ax = plt.subplots()
    _frame(ax, x_train, y_train, x_val, y_val)
    ax.legend(loc="upper right", frameon=False)
    plt.show()


def plot_fit(p, x_train, y_train, x_val, y_val, label):
    fig, ax = plt.subplots()
    _frame(ax, x_train, y_train, x_val, y_val)
    ax.plot(XX, p(XX), color=YELLOW, lw=2.5, label=label)
    ax.legend(loc="upper right", frameon=False)
    plt.show()


def plot_errors(degrees, train_err, val_err, best=None):
    fig, ax = plt.subplots()
    ax.plot(degrees, train_err, "o-", color=INK, label="training error")
    ax.plot(degrees, val_err, "D-", color=LIME, label="validation error")
    ax.axhline(SIG ** 2, color=GREY, ls=":", lw=1.2)
    ax.text(degrees[-1], SIG ** 2 * 1.2, r"noise level $\sigma^2 = 0.09$", color=GREY, fontsize=10, ha="right")
    if best is not None:
        ax.scatter([best], [val_err[list(degrees).index(best)]], s=260, facecolors="none",
                   edgecolors=INK, linewidths=2, zorder=4, label=f"best: degree {best}")
    ax.set_yscale("log")
    ax.set(xlabel="polynomial degree", ylabel="mean squared error (log scale)", xticks=degrees)
    ax.legend(frameon=False, loc="upper left")
    plt.show()


def show_fits(x_train, y_train, x_val, y_val, degrees):
    fig, axs = plt.subplots(1, len(degrees), figsize=(4 * len(degrees), 3.4), sharey=True)
    for ax, d in zip(axs, degrees):
        p = Polynomial.fit(x_train, y_train, deg=d)
        _frame(ax, x_train, y_train, x_val, y_val)
        ax.plot(XX, p(XX), color=YELLOW, lw=2.5)
        err = np.mean((p(x_val) - y_val) ** 2)
        ax.set_title(f"degree {d}: validation error {err:.3g}", fontsize=11)
    axs[0].legend(loc="lower left", frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_grid(search):
    """Heat map of the cross-validated error of a GridSearchCV over degree and ridge__alpha."""
    res = search.cv_results_
    degs = np.array(res["param_polynomialfeatures__degree"], dtype=float)
    lams = np.array(res["param_ridge__alpha"], dtype=float)
    D, L = np.unique(degs), np.unique(lams)
    E = np.full((len(D), len(L)), np.nan)
    for d, l, e in zip(degs, lams, -res["mean_test_score"]):
        E[np.searchsorted(D, d), np.searchsorted(L, l)] = e
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    im = ax.imshow(E, origin="lower", aspect="auto", cmap="viridis",
                   norm=LogNorm(vmin=np.nanmin(E), vmax=min(np.nanmax(E), 1.0)))
    ax.set_xticks(range(len(L)), [f"{l:.0e}" for l in L], rotation=45)
    ax.set_yticks(range(len(D)), [f"{d:.0f}" for d in D])
    ax.set(xlabel=r"$\lambda$ (ridge__alpha)", ylabel="polynomial degree")
    b = search.best_params_
    ax.scatter([np.searchsorted(L, b["ridge__alpha"])], [np.searchsorted(D, b["polynomialfeatures__degree"])],
               s=200, facecolors="none", edgecolors="white", linewidths=2.5)
    fig.colorbar(im, ax=ax, label="5-fold CV error (log scale)")
    plt.tight_layout()
    plt.show()


def plot_test(model, X_dev, y_dev, X_test, y_test, label):
    fig, ax = plt.subplots()
    _frame(ax, X_dev[:, 0], y_dev, X_test[:, 0], y_test, other="test", train_label="development")
    ax.plot(XX, model.predict(XX[:, None]), color=YELLOW, lw=2.5, label=label)
    ax.legend(loc="upper right", frameon=False, fontsize=9)
    plt.show()


def plot_validation_curve(lams, train_err, val_err):
    fig, ax = plt.subplots()
    ax.plot(lams, train_err, "o-", color=INK, label="training error")
    ax.plot(lams, val_err, "D-", color=LIME, label="5-fold CV error")
    k = np.argmin(val_err)
    ax.scatter([lams[k]], [val_err[k]], s=260, facecolors="none", edgecolors=INK, linewidths=2, zorder=4)
    ax.set_xscale("log")
    ax.set(xlabel=r"penalty $\lambda$ (log scale)", ylabel="mean squared error")
    ax.legend(frameon=False)
    plt.show()


def plot_scores(scores):
    """Horizontal bars for a dict {description: balanced accuracy}."""
    names, vals = list(scores), list(scores.values())
    fig, ax = plt.subplots(figsize=(7, 0.6 * len(names) + 1))
    ax.barh(range(len(names)), vals, color=[MAGENTA if v > 0.75 else INK for v in vals], height=0.6)
    for i, v in enumerate(vals):
        ax.text(v + 0.01, i, f"{v:.0%}", va="center")
    ax.axvline(0.5, color=GREY, ls="--", lw=1.2)
    ax.text(0.505, -0.55, "chance", color=GREY, fontsize=10, va="bottom")
    ax.set_yticks(range(len(names)), names)
    ax.set(xlim=(0, 1.05), ylim=(len(names) - 0.5, -0.8), xlabel="cross-validated balanced accuracy")
    plt.tight_layout()
    plt.show()

---
## ★ Core · Overfitting by hand

On the slides, polynomials of rising degree drew a U: the training error falls forever, the test error falls and then climbs. Here you build that figure yourself. The data come from the same process as on the slides: $x$ uniform in $[0, 1]$, and $y = \sin(2\pi x)$ plus Gaussian noise with $\sigma = 0.3$.

### Exercise 1 · Two sets, two jobs

*Slide: Split Your Data.* We draw 40 points and hide half of them from the model: 20 points to fit (as many as on the slides), 20 to validate. `train_test_split` shuffles the points and cuts them in two; its argument `test_size` is the fraction that goes into the second set. (We will meet the third set, the test set, in the Stretch part.)

Fill in the blank so that half the points go to validation.

In [ ]:
x, y = sample(40, seed=SEED)
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=____, random_state=SEED)

assert len(x_train) == 20 and len(x_val) == 20, "You should have 20 training and 20 validation points: check test_size."
print("Checkpoint passed: 20 training points, 20 validation points.")
plot_data(x_train, y_train, x_val, y_val)

**Question 1.** `sample` returns the points sorted by $x$. Why not simply take the first 20 points for training and the last 20 for validation?

*Your answer:* 

### Exercise 2 · One fit, two errors

*Slide: Two Kinds of Error.* `Polynomial.fit(x, y, deg=d)` finds the polynomial of degree `d` with the smallest squared error on `(x, y)`: yesterday's loss, solved exactly instead of by gradient descent. The result `p` is a function: `p(x)` gives the predictions at `x`.

Write the mean squared error, then measure the cubic on both sets.

In [ ]:
def mse(y_true, y_pred):
    """Mean squared error: the average of the squared residuals."""
    # YOUR CODE HERE
    # HINT: square the residuals y_true - y_pred, then average them with np.mean
    ...


p = Polynomial.fit(x_train, y_train, deg=3)
err_train = mse(y_train, p(x_train))
err_val = mse(y_val, ____)
print(f"degree 3: training error {err_train:.3f}, validation error {err_val:.3f}")

assert np.isclose(mse(np.array([1.0, 2.0]), np.array([1.0, 4.0])), 2.0), "mse([1, 2], [1, 4]) should be 2."
assert err_train < err_val < 0.3, "The validation error should be a bit larger than the training error, and well below 0.3: did you predict at x_val?"
print("Checkpoint passed.")
plot_fit(p, x_train, y_train, x_val, y_val, label="degree 3")

You should see a training error of 0.082 and a validation error of 0.108.

**Question 2.** The noise has variance $\sigma^2 = 0.3^2 = 0.09$. Why can no model, not even the true function $\sin(2\pi x)$, reach a validation error much below 0.09 on average? What would a *training* error far below 0.09 tell you?

*Your answer:* 

### Exercise 3 · Every degree from 1 to 15

*Slide: The U-Curve.* Now repeat the fit for every degree, and record both errors each time.

In [ ]:
degrees = np.arange(1, 16)
train_err, val_err = [], []
for d in degrees:
    p = Polynomial.fit(x_train, y_train, deg=____)
    # YOUR CODE HERE
    # HINT: append the training error of p to train_err, and its validation error to val_err
    ...
train_err, val_err = np.array(train_err), np.array(val_err)

assert len(train_err) == len(val_err) == 15, "You need one training and one validation error per degree: 15 of each."
assert np.all(np.diff(train_err) <= 1e-9), "The training error should never go up with the degree: check which points you used."
print("Checkpoint passed.\n")
print("degree   training   validation")
for d, tr, va in zip(degrees, train_err, val_err):
    print(f"{d:6d}   {tr:8.3f}   {va:10.3g}")

### Exercise 4 · The U-curve, and the best degree

Plot both errors against the degree (the vertical axis is logarithmic, because the validation error grows by orders of magnitude). Then choose the degree with the smallest **validation** error: `np.argmin` gives the position of the smallest value in an array.

In [ ]:
best = degrees[np.argmin(____)]
print(f"Best degree on the validation set: {best}")

assert 2 <= best <= 6, "The best degree should be a small one. Which error are you minimising?"
print("Checkpoint passed.")
plot_errors(degrees, train_err, val_err, best)
show_fits(x_train, y_train, x_val, y_val, [1, best, 15])

You should see the U of the slides: degree 1 underfits (both errors high), degree 3 is best, and beyond degree 8 the validation error explodes while the training error keeps going down. In the right-hand panel, the degree-15 curve passes close to every black point and swings wildly in between.

**Question 3.** Why does the training error keep falling as the degree grows, all the way to degree 15? Why does that make it useless for choosing the degree?

*Your answer:* 

### Exercise 5 · One split, one opinion

*Slide: Cross-Validation.* Your best degree came from **one** random split. Run the cell below: it repeats Exercises 1 to 4 for five other splits of the same 40 points (only `random_state` changes).

In [ ]:
for rs in range(1, 6):
    xt, xv, yt, yv = train_test_split(x, y, test_size=0.5, random_state=rs)
    errs = [mse(yv, Polynomial.fit(xt, yt, deg=d)(xv)) for d in degrees]
    print(f"split {rs}: best degree {degrees[np.argmin(errs)]}, its validation error {min(errs):.3f}")

**Question 4.** What changes from split to split, and what stays the same? What does that say about reporting the validation error of a single split?

*Your answer:* 

**End of the Core.** If you have time left, go on: the Stretch part does the same job properly, the way you would in a real project.

---
## ★★ Stretch · Cross-validation with a pipeline

The full protocol from the slides: split off a test set **first** and lock it away; choose the settings by 5-fold cross-validation on the rest; then test once. And instead of choosing only the degree, we use ridge regression (*Penalise Large Weights*) and choose the degree and the penalty λ together. Fewer hints from here on.

A scikit-learn **pipeline** chains the steps into one model, so that cross-validation refits every step on each training fold (*The Fix: Pipelines*):

1. `PolynomialFeatures(degree=15, include_bias=False)` turns each $x$ into the columns $x, x^2, \dots, x^{15}$ (no column of ones: the ridge has its own intercept $b$);
2. `StandardScaler()` gives every column mean 0 and standard deviation 1, so that the penalty treats all weights alike;
3. `Ridge(alpha=lam)` fits the weights $w$ and the intercept $b$ by least squares plus $\lambda \sum_j w_j^2$. scikit-learn calls λ `alpha`.

### Exercise 6 · Lock away the test set, then cross-validate

New data from the same process: 100 points, of which 20% become the test set. scikit-learn wants a 2-D array of inputs, one row per point and one column per feature, so `X = x[:, None]`.

Then cross-validate the degree-15 pipeline for the three λ of the slide *Turning the Knob λ*. scikit-learn always *maximises* a score, so it reports errors with a minus sign: use `scoring="neg_mean_squared_error"` and flip the sign.

In [ ]:
x, y = sample(100, seed=SEED + 1)
X = x[:, None]
# YOUR CODE HERE
# HINT: X_dev, X_test, y_dev, y_test = train_test_split(...), with 20% for the test set and random_state=SEED
...
assert len(y_test) == 20 and len(y_dev) == 80, "You should have 80 development points and 20 test points."

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
for lam in [1e-10, 1e-3, 1.0]:
    # YOUR CODE HERE
    # HINT: pipe = make_pipeline(three steps, the last with alpha=lam)
    # HINT: scores = cross_val_score(pipe, X_dev, y_dev, cv=cv, scoring=...)
    ...
    print(f"lambda = {lam:<6g}  CV error {-scores.mean():.3f}   (the 5 folds: {np.round(-scores, 3)})")

assert len(scores) == 5, "cross_val_score should return one score per fold: 5."
print("\nCheckpoint passed.")

You should see the order of the slide: λ ≈ 0 overfits (0.120), λ = 0.001 is best (0.095), λ = 1 underfits (0.153). The differences are smaller than on the slide because we now fit 64 points per fold instead of 20: more data is a regulariser too.

**Question 5.** Why must the `StandardScaler` sit *inside* the pipeline, instead of being applied once to `X_dev` before calling `cross_val_score`?

*Your answer:* 

### Exercise 7 · Choose the degree and λ together

*Slide: Choosing the Degree.* `GridSearchCV` cross-validates every combination of settings in a grid and keeps the best. Inside a pipeline, a setting is called `step__parameter`, with **two** underscores and the step's name in lower case: `polynomialfeatures__degree` and `ridge__alpha`.

Your design decision: which degrees and which λ to try. A good start is every degree from 1 to 15, and λ from $10^{-8}$ to $10^{2}$ in factors of ten (`np.logspace(-8, 2, 11)`).

In [ ]:
# YOUR CODE HERE
# HINT: param_grid = {"step__parameter": values to try, ...}, with two entries
...
search = GridSearchCV(pipe, param_grid, cv=cv, scoring="neg_mean_squared_error")
search.fit(X_dev, y_dev)

best_degree, best_lam = search.best_params_["polynomialfeatures__degree"], search.best_params_["ridge__alpha"]
print(f"Best setting: degree {best_degree}, lambda {best_lam:g}; its CV error {-search.best_score_:.3f}")
assert -search.best_score_ < 0.15, "The best CV error should be close to the noise level 0.09. Check your grid."
print("Checkpoint passed.")
plot_grid(search)

With the suggested grid you should see degree 15 with λ = 0.01, and a CV error of 0.095, close to the noise floor.

**Question 6.** Where on the map is the CV error high, and why? Is the winner much better than its neighbours?

*Your answer:* 

### Exercise 8 · Test once

`GridSearchCV` has already refitted the best pipeline on all of `X_dev`, and `search.predict` uses it. Now, and only now, unlock the test set. Compute the test error, and compare it with the CV error, with a trivial baseline that always predicts the mean of `y_dev` (*Always Compare to a Baseline*), and with the noise floor.

In [ ]:
# YOUR CODE HERE
# HINT: test_err = mean_squared_error(true values, predictions), with search.predict on the test inputs
...
baseline = DummyRegressor(strategy="mean").fit(X_dev, y_dev)
base_err = mean_squared_error(y_test, baseline.predict(X_test))

print(f"CV error (development set)  {-search.best_score_:.3f}")
print(f"test error                  {test_err:.3f}")
print(f"baseline: always the mean   {base_err:.3f}")
print(f"noise floor, sigma^2        {SIG**2:.3f}")
assert test_err < base_err, "Your model should beat the baseline by a wide margin."
print("Checkpoint passed.")
plot_test(search.best_estimator_, X_dev, y_dev, X_test, y_test,
          label=f"degree {best_degree}, lambda {best_lam:g}")

**Question 7.** The test error is a little higher than the CV error. Why is that expected? And why must you not now go back, change the grid, and try again to lower the test error?

*Your answer:* 

### Exercise 9 (optional) · The validation curve for λ

`validation_curve` cross-validates one pipeline for a list of values of a single setting, and returns the training and the validation scores for each value and fold. Draw it for λ, at degree 15. Name the setting as in the grid search.

In [ ]:
lams = np.logspace(-8, 2, 11)
pipe15 = make_pipeline(PolynomialFeatures(degree=15, include_bias=False), StandardScaler(), Ridge())
train_scores, val_scores = validation_curve(pipe15, X_dev, y_dev, param_name=____, param_range=lams,
                                            cv=cv, scoring="neg_mean_squared_error")
plot_validation_curve(lams, -train_scores.mean(axis=1), -val_scores.mean(axis=1))

You should see the training error rise steadily with λ, and the CV error dip to its minimum around λ = 0.001–0.01 before climbing steeply.

**Question 8.** Compare this plot with your U-curve from Exercise 4. Which side is "too flexible" here, and which "too simple"?

*Your answer:* 

---
## ★★★ Challenge · Find the leak

A colleague in a neuroscience lab sends you an exciting result. They imaged 200 neurons in mouse cortex over 12 sessions, one per day, 40 trials per session. On each trial the mouse either licked the water spout or not. Their decoder predicts, from the activity of the neurons on a **single trial**, whether the mouse licked, with a cross-validated balanced accuracy of almost 90%. The paper draft is ready.

You are the data curator. Before the draft goes out, check the number. (Work in pairs if you like: two curators argue better than one.)

*Balanced accuracy* is the average of the recall of the two classes (*Precision and Recall*): the fraction of lick trials the decoder catches, averaged with the fraction of no-lick trials it catches. A model that ignores its input scores 50%, whatever the class balance.

Run the two cells below: the first loads the data, the second is your colleague's analysis, exactly as they sent it.

In [ ]:
# @title Load the lab's recordings (run this cell; please do not read it before you have finished) { display-mode: "form" }
def record_sessions(seed):
    """Simulated imaging data: 12 sessions (one per day) x 40 trials, 200 neurons.
    Returns X (trials x neurons, activity), y (1 = licked, 0 = did not) and session (0 to 11)."""
    rng = np.random.default_rng(seed)
    n_sessions, n_trials, n_neurons = 12, 40, 200
    thirst = rng.permutation([0.95, 0.95, 0.9, 0.9, 0.9, 0.85, 0.15, 0.1, 0.1, 0.1, 0.05, 0.05])
    session = np.repeat(np.arange(n_sessions), n_trials)
    y = (rng.random(session.size) < thirst[session]).astype(int)        # thirsty days: licks on most trials
    day_baseline = rng.normal(0, 0.5, (n_sessions, n_neurons))          # new day, new field of view, new baseline
    w = np.zeros(n_neurons)
    w[rng.choice(n_neurons, 20, replace=False)] = 0.5                   # 20 neurons fire a little more on licks
    X = day_baseline[session] + np.outer(y, w) + rng.normal(0, 1, (session.size, n_neurons))
    order = rng.permutation(session.size)                               # the lab's export shuffles the trials
    return X[order], y[order], session[order]


X, y, session = record_sessions(seed=SEED + 2)
print(f"X: {X.shape[0]} trials x {X.shape[1]} neurons;  y: 1 = licked, 0 = did not;  session: the recording session of each trial")
print("first ten trials:  y =", y[:10], "  session =", session[:10])

In [ ]:
# Your colleague's analysis
X_scaled = StandardScaler().fit_transform(X)                             # z-score every neuron
X_best = SelectKBest(f_classif, k=20).fit_transform(X_scaled, y)         # keep the 20 most lick-related neurons
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(LogisticRegression(max_iter=1000), X_best, y, cv=cv, scoring="balanced_accuracy")
print(f"Decoding licks from single trials: balanced accuracy {scores.mean():.0%} (folds: {np.round(scores, 2)})")

### Exercise 10 · Investigate

Your task: find out why this number is too good, compute an honest one, and write your colleague a short report. Nobody will tell you where the problem is; the checklist from the slides is your guide. For each item that applies, write a **probe**: a few lines of code whose output would look different if the result were honest. (There are two leaks: a big one and a smaller one.)

**A curator's checklist** (*slide: A Curator's Checklist*)
1. Split first; test once.
2. Split by animal, session, patient.
3. Look for duplicates.
4. Fit preprocessing inside the folds.
5. Compare to a baseline.
6. Read the confusion matrix, not just the accuracy.
7. Ask where the next data will come from.

**Toolbox.** Everything is already imported.
- `DummyClassifier()`: a classifier that ignores `X` and always predicts the most frequent class of its training data.
- `GroupKFold(n_splits=5)`: like `KFold`, but all rows with the same group label land in the same fold; pass the labels with `cross_val_score(..., groups=...)`.
- `make_pipeline(step1, step2, ..., model)`: chains steps into one model that cross-validation refits on every training fold.
- `np.unique(a)`, a boolean mask such as `y[session == 3]`, and `.mean()` for quick counts. You can also ask a classifier to predict something other than `y`.

Stuck? Open the hints one at a time.

<details><summary>Hint 1</summary>

Start with items 5 and 2. What does a model that ignores the neurons score? What is the unit of this dataset, the thing of which a new one will come along? Do the labels look the same in every unit?

</details>

<details><summary>Hint 2</summary>

Item 7: the decoder is meant to work on tomorrow's session. In your colleague's cross-validation, does any test trial come from a session the model has never seen? And can the neurons tell you which session a trial comes from? Try predicting `session` instead of `y`.

</details>

<details><summary>Hint 3</summary>

Item 4: read the first two lines of the analysis again. Which step looks at `y` before the data are split into folds?

</details>

<details><summary>Hint 4</summary>

The honest evaluation needs both fixes at once: a pipeline (scaler, selector, classifier) cross-validated with `GroupKFold`, with `groups=session`.

</details>

Use the cells below for your probes (add more if you like).

In [ ]:
# YOUR CODE HERE
# HINT: a probe is a few lines whose output you can compare with the colleague's number
...

In [ ]:
# YOUR CODE HERE
# HINT: another probe
...

In [ ]:
# YOUR CODE HERE
# HINT: another probe
...

In [ ]:
# YOUR CODE HERE
# HINT: another probe (or leave this cell empty)
...

**Question 9.** What did your probes show? Explain, in a few sentences, where the 90% comes from.

*Your answer:* 

### Exercise 11 · The honest number

Fix what you found, and compute the balanced accuracy your colleague should report. Compare it with the baseline, evaluated the same way.

In [ ]:
# YOUR CODE HERE
# HINT: honest = cross_val_score(...): the colleague's three steps, evaluated without the leaks you found
...
print(f"honest balanced accuracy: {honest.mean():.0%} (folds: {np.round(honest, 2)})")
assert honest.mean() < 0.75, "Still above 75%: at least one of the two leaks is still open."
assert honest.mean() > 0.55, "Below 55%: check that you still use the colleague's three steps."
print("Checkpoint passed.")

For the report it helps to know how much each leak contributes on its own. Fix one at a time, and draw all your numbers with `plot_scores`, which takes a dictionary `{description: score}`.

In [ ]:
# YOUR CODE HERE
# HINT: one cross_val_score per combination of fixes, then plot_scores({...})
...

**Question 10.** Write your colleague a three-line report: the number they should publish, what was wrong, and one sentence on what the honest number means.

*Your answer:* 

---
## Wrap-up

- **What you just saw.** The training error always falls with flexibility; only data the model has not seen draws the U and picks the degree. Cross-validation turns one noisy validation split into an average, ridge's λ tames a flexible model, and the test set is opened once.
- **The curator's lesson.** A leak can turn a weak signal into 88%. Split by the unit that will be new (session, animal, patient), keep every step that touches the data inside the folds with a pipeline, and compare with a baseline under the same split.
- **Where it comes back.** On Thursday, *Beyond the Straight Line*: k-nearest neighbours, SVMs, trees and forests, each with its own flexibility knob (k, C, depth), chosen exactly as you chose the degree and λ today. In Notebook 3 you will decode real recordings from mouse visual cortex with the same tools: a pipeline, cross-validation, a baseline and a confusion matrix.